# A bounded vision pipeline

The capstone joins preprocessing, a detector, crop filtering, classification, and a JSON-serializable result. All inputs are local NumPy fixtures.

In [ ]:
from pathlib import Path
import importlib.util
import sys
lesson_rel = Path('phases/04-computer-vision/16-vision-pipeline-capstone')
candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base / lesson_rel / 'code/main.py', base / 'code/main.py'))
code_path = next(p.resolve() for p in candidates if p.is_file())
spec = importlib.util.spec_from_file_location('cv04_l16_nb', code_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
assert spec.loader is not None
spec.loader.exec_module(module)
print(code_path)

In [ ]:
if not module.TORCH_AVAILABLE:
    import numpy as np
    result = module.numpy_pipeline(np.zeros((64, 96, 3), dtype=np.uint8), image_id='notebook-fixture')
    payload = result.to_dict()
    assert len(payload['detections']) == 3 and len(payload['classifications']) == 3
    print({'Build-It': 'NumPy pipeline', 'detections': len(payload['detections']), 'classifications': len(payload['classifications']), 'Use-It': 'PyTorch skipped cleanly'})
else:
    import numpy as np
    pipe = module.VisionPipeline(module.StubDetector(), module.StubClassifier(3), ['background', 'object', 'other'])
    result = pipe.run(np.zeros((48, 64, 3), dtype=np.uint8), image_id='notebook-fixture')
    payload = result.to_dict()
    assert payload['image_id'] == 'notebook-fixture' and 'detections' in payload
    print(result.to_json())

A real service would add transport and authentication outside this lesson. The local acceptance check is the stable result schema and the explicit crop policy.